# Multi Query Retriever — 쿼리 다각화

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- 단일 쿼리 검색의 한계와 **MultiQueryRetriever**가 이를 보완하는 원리 이해하기
- `MultiQueryRetriever.from_llm()`으로 LLM 기반 쿼리 다각화 구현하기
- 로깅으로 LLM이 생성한 쿼리 목록을 직접 확인하기

## 사전 지식

- VectorStoreRetriever 기본 사용법
- LLM이 텍스트를 이해하고 변환하는 원리

---

## 핵심 아이디어

하나의 쿼리를 LLM이 여러 관점으로 재구성해요:

| | 내용 |
|---|---|
| **원본 쿼리** | "딥러닝이 뭐야?" |
| **변환 쿼리 1** | "딥러닝의 정의는 무엇인가?" |
| **변환 쿼리 2** | "딥러닝 기술에 대해 설명해주세요" |
| **변환 쿼리 3** | "딥러닝은 어떤 학습 방법인가?" |

각 쿼리로 검색 후 결과를 통합(중복 제거)합니다.

> **주의**: MultiQueryRetriever는 LLM을 추가로 호출하기 때문에 응답 속도가 느려지고 비용이 증가해요. 검색 품질이 중요한 경우에 선택적으로 사용하세요.

> **실무 팁**: 로깅 레벨을 INFO로 설정하면 LLM이 생성한 쿼리들을 콘솔에서 확인할 수 있어요.

In [3]:
from dotenv import load_dotenv
load_dotenv()


True

In [5]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ---------------------------------------------------
# MultiQueryRetriever를 위한 기본 Retriever와 LLM 준비
# ---------------------------------------------------

# ============================================================
# TODO: ai-story.txt 기반 FAISS Retriever와 ChatOpenAI LLM을 생성하세요
# 힌트: base_retriever는 k=3, LLM은 temperature=0 (일관된 쿼리 생성)
# 예상 결과: Retriever와 LLM 준비 완료 메시지 출력
# ============================================================

# 문서 및 retriever 준비


# temperature=0: 쿼리 생성 결과를 일관되게 유지
loader = TextLoader("./data/ai-story.txt")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(split_docs, embeddings)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")


In [6]:
# ---------------------------------------------------
# MultiQueryRetriever 생성 및 LLM 생성 쿼리 확인
# ---------------------------------------------------

# ============================================================
# TODO: MultiQueryRetriever.from_llm()으로 생성하고 INFO 로그로 생성된 쿼리를 확인하세요
# 힌트: logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)
# 예상 결과: LLM이 생성한 여러 쿼리가 로그에 출력되고 문서가 검색됨
# ============================================================

from langchain.retrievers.multi_query import MultiQueryRetriever
import logging

# 로깅 설정 — INFO 레벨로 LLM이 생성한 쿼리 목록 확인 가능
# 로깅 설정 — INFO 레벨로 LLM이 생성한 쿼리 목록 확인 가능
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

# MultiQueryRetriever: 원본 쿼리를 LLM이 3~5개로 재구성

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm
)

query = "NLP에서 임베딩은 어떤 역할을 하나요?"
docs = multi_query_retriever.invoke(query)

docs


INFO:langchain.retrievers.multi_query:Generated queries: ['NLP에서 임베딩의 기능은 무엇인가요?  ', '자연어 처리에서 임베딩이 수행하는 역할은 무엇인지 알고 싶습니다.  ', 'NLP에서 임베딩 기술이 중요한 이유는 무엇인가요?']


[Document(id='c7dc19e5-2478-46d9-b1b3-1b629ef4c8f3', metadata={'source': './data/ai-story.txt'}, page_content='NLP의 핵심 과제 중 하나는 자연어의 모호성과 다양성을 처리하는 것입니다. 인간 언어는 복잡하고, 문맥에 따라 그 의미가 달라질 수 있으며, 같은 단어나 구문이 다양한 의미로 사용될 수 있습니다. 이러한 언어의 특성으로 인해, NLP 시스템은 텍스트의 문법적 구조를 파악하고, 문맥을 분석하여 의미를 정확하게 이해할 수 있어야 합니다. 이를 위해 NLP는 통계학, 기계 학습, 딥 러닝과 같은 다양한 수학적 및 계산적 방법론을 활용합니다.\n\n최근 몇 년 동안, 딥 러닝 기반의 NLP 모델, 특히 변환기(Transformer) 아키텍처와 같은 신경망 모델의 발전으로 인해, 자연어 이해와 생성 분야에서 놀라운 진보가 이루어졌습니다. 이러한 모델들은 대규모의 언어 데이터에서 학습하여, 문장의 문맥을 효과적으로 파악하고, 보다 정교한 언어 이해 및 생성 능력을 제공합니다.'),
 Document(id='1deceb97-a9b1-475c-9322-dce452860ea7', metadata={'source': './data/ai-story.txt'}, page_content='NLP의 발전은 사람과 컴퓨터 간의 소통 방식을 근본적으로 변화시켰습니다. 예를 들어, 자연어를 이해할 수 있는 인터페이스를 통해 사용자는 복잡한 쿼리나 명령어를 사용하지 않고도 정보를 검색하거나 서비스를 이용할 수 있게 되었습니다. 또한, 자동 번역 시스템의 발전으로 언어 장벽이 낮아지고 있으며, 감정 분석 기술을 통해 소셜 미디어나 고객 리뷰에서 유용한 인사이트를 추출할 수 있게 되었습니다.\n\nNLP는 계속해서 발전하고 있으며, 이 분야의 연구와 기술 혁신은 인간 언어의 복잡성을 더 깊이 이해하고, 인간과 기계 간의 상호작용을 더욱 풍부하고 의미 있게 만드는 데 기여할 것입니다.\n\nScipy

---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 아이디어 | LLM이 원본 쿼리 → 여러 관점의 쿼리를 자동 생성, 결과 합산 |
| 장점 | 단일 쿼리 한계 극복, 다양한 표현으로 recall(재현율) 향상 |
| 단점 | LLM 추가 호출 비용 발생, 응답 지연 증가 |
| 중복 제거 | 내부적으로 중복 문서를 자동으로 제거해줘요 |
| 로깅 확인 | `logging.getLogger("langchain...").setLevel(logging.INFO)` |

### 생성 쿼리 예시

원본 쿼리: "머신러닝에서 과적합을 방지하는 방법은?"

| 생성 쿼리 | 관점 |
|-----------|------|
| 과적합 문제를 해결하기 위한 정규화 기법은? | 기법 중심 |
| 딥러닝 모델의 일반화 성능을 높이는 전략은? | 목적 중심 |
| Train/Test 성능 차이를 줄이는 방법은? | 증상 중심 |

### 다음 노트북 예고

**07-MultiVectorRetriever** — 원본 문서 대신 요약본이나 가상 질문으로 임베딩하여 검색하고, 실제 반환은 원본 문서를 돌려주는 전략을 배워요.